In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Custom Layers and Functions

A framework's library holds the layers that many people needed yesterday:
fully connected layers, activation functions, dropout. Research moves
faster than libraries, and a layer that a paper proposes can be missing
from them for years. RMSNorm, a layer that rescales each activation
vector to a fixed size, is an example. It was published in 2019
[@Zhang.Sennrich.2019], and PyTorch added it to its library of layers
in 2024. Until then, each PyTorch codebase that used it carried its own
implementation or borrowed one from another package. How do we write such a
layer ourselves, so that it behaves like a built-in one? Its parameters must
reach the optimizer, its state must be saved and moved to the GPU, and it
must fit into containers next to other layers.

Sometimes the trouble lies with the derivative instead. To fit a large
model into a small memory, its weights are often stored as 4-bit integers:
the weights of the 8B model of that section then take
4 GB, a quarter of what 16-bit floating-point numbers need. Storing
integers means rounding, and rounding has derivative zero almost
everywhere, so a weight that is rounded in the forward pass receives no
gradient and never learns.

This section writes both kinds of extension. A custom layer is a subclass
of the module class of that section. Everything the
framework does to a whole model, from collecting parameters for the
optimizer to saving a checkpoint, walks the registered state, so a layer
that registers its state needs no further code to take part. The ideas
here are few. Most of the section shows the form in which each framework
expects them. We start with a layer without state, then write RMSNorm with
a learned parameter, then a layer with fixed state that must be saved but
never trained. Finally we replace the derivative of an operation with a
rule of our own, which lets gradients pass through rounding.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Layers without Parameters

Let us start with the smallest custom layer, one with no state at all.
`CenteredLayer` subtracts from its input the mean over all of its entries,
the batch axis included (an exercise asks what that implies). It inherits
from the module class and supplies only the forward method; there is
nothing to set up.

In [ ]:
class CenteredLayer(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, X):
        return X - tf.reduce_mean(X)

The inputs 1 to 5 have mean 3, so the output should run from $-2$ to $2$:

In [ ]:
layer = CenteredLayer()
layer(tf.constant([1.0, 2, 3, 4, 5]))

The class has the same interface as a built-in layer, so a sequential
container can call it like any other child. Placed after a linear layer, it
centers the linear layer's outputs. Their mean should be zero. In
floating-point arithmetic it comes out as exactly zero only by luck; a
number of size $10^{-8}$ or smaller is rounding error and no cause for
alarm.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(128), CenteredLayer()])
Y = net(tf.random.uniform((4, 8)))
tf.reduce_mean(Y)

## Layers with Parameters

In that section we saw that the scale of the
activations can grow or shrink from one layer to the next, by a factor that
depends on the weights. A *normalization layer* removes this dependence: it
rescales each input vector to a fixed size and lets a learned parameter set
the size instead. Such layers form a family that we study from
that section onwards. Here we implement its simplest member,
*RMSNorm*.

### RMSNorm

The root mean square of a vector $\mathbf{x} \in \mathbb{R}^d$ is
$\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}$. RMSNorm divides each example's
vector by it and multiplies the result elementwise by a learned *gain*
$\mathbf{g} \in \mathbb{R}^d$:

$$
\textrm{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}} \odot \mathbf{g}.
$$

The small constant $\epsilon$, such as $10^{-6}$, keeps the denominator away
from zero. With the gain at its initial value $\mathbf{g} = \mathbf{1}$, the
output has root mean square close to 1, whatever the scale of the input.
For $\mathbf{x} = (3, 4)$ the mean square is $(9 + 16)/2 = 12.5$, the root
mean square is $3.536$, and the output is $(0.849, 1.131)$, whose mean
square is 1. The input $(300, 400)$ gives the same output.
the figure shows the geometry: RMSNorm keeps the direction of
$\mathbf{x}$, discards its length by moving it onto the sphere of radius
$\sqrt{d}$, and the gain then stretches each coordinate.

![RMSNorm in two dimensions. Left: with gain 1, every input (blue) moves along its ray from the origin onto the circle of radius $\sqrt{2}$ (orange); two inputs on the same ray, at lengths 0.6 and 2.9, land on the same point. Right: the gain $\mathbf{g} = (1.5, 0.5)$ stretches the circle into an ellipse with semi-axes $1.5\sqrt{2}$ and $0.5\sqrt{2}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-rmsnorm.svg)

RMSNorm simplifies *layer normalization* [@Ba.Kiros.Hinton.2016],
which first subtracts the mean of the entries of each example's vector,
as a per-example `CenteredLayer` would, and then adds a learned shift after
the gain. Zhang and Sennrich found that the centering contributes little to
training, and many recent language models use RMSNorm, among them Llama 3
8B [@Grattafiori.Dubey.Jauhri.ea.2024], the 8B model of
that section.

The gain is to be learned, so the layer must register it as a parameter
(that section). Apart from that, the code below follows the
formula.

In Keras, `add_weight` creates and registers a variable. We call it in the
method `build`, which runs on the first call once the input shape is known,
so our layer need not be told its width.

In [ ]:
class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def build(self, X_shape):
        self.gain = self.add_weight(name='gain', shape=[X_shape[-1]],
                                    initializer='ones')

    def call(self, X):
        rms = tf.sqrt(
            tf.reduce_mean(X ** 2, axis=-1, keepdims=True) + self.eps)
        return self.gain * X / rms

We try the layer on the worked example and on the same vector scaled by
100. Both rows should read $(0.849, 1.131)$:

In [ ]:
norm = RMSNorm()
norm(tf.constant([[3.0, 4.0], [300.0, 400.0]]))

### Registration

Why did we write RMSNorm as a module and not as a function with a gain
tensor kept beside it? Because registration makes a custom layer behave
like a built-in one in four ways: its parameters are found, it works inside
containers, its state is saved and loaded, and it moves between devices.
None of this is new. that section and
that section showed all four for the library's layers, so one
check on a layer of our own suffices. We place RMSNorm between two linear
layers, take one optimizer step, which moves the gain away from its initial
ones, and copy the state of the model into a second model with the same
structure and its own random initial weights. Afterwards the two compute the
same outputs (that section writes such state to disk).

An optimizer given `net.trainable_variables` finds the gain and updates it.
`get_weights` returns the model's weights, the gain included, as a list of
arrays, and `set_weights` loads the list into the clone once a first call
has built it. Keras places the gain where `add_weight` creates it, on a GPU
if one is visible, like any built-in weight (that section).

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(8), RMSNorm(),
                           tf.keras.layers.Dense(2)])
X = tf.random.normal((4, 20))
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(net(X) ** 2)
grads = tape.gradient(loss, net.trainable_variables)
tf.keras.optimizers.SGD(0.1).apply_gradients(
    zip(grads, net.trainable_variables))
clone = tf.keras.Sequential([tf.keras.layers.Dense(8), RMSNorm(),
                             tf.keras.layers.Dense(2)])
clone(X)  # A first call builds the clone so its weights exist
clone.set_weights(net.get_weights())
net.layers[1].gain.numpy(), bool(tf.reduce_all(net(X) == clone(X)))

The gain is no longer all ones, and the clone computes exactly the outputs
of the trained model. None of this required any code in `RMSNorm` beyond the
registration of its gain.

### Comparison with Framework Implementations

The checks so far say that the layer is a good citizen of the framework.
They do not say that it computes the right thing. Where the library
provides the same layer, we can find out by comparing the outputs of the
two implementations.

Keras's `RMSNormalization` calls its gain `scale` and its divisor guard
`epsilon`. We give both implementations the same gain, other than all ones,
and compare their outputs; once both layers are built, `get_weights` and
`set_weights` copy the gain across.

In [ ]:
ours, native = RMSNorm(), tf.keras.layers.RMSNormalization(epsilon=1e-6)
X = tf.random.normal((16, 8))
ours(X)  # Build both so the weights exist
native.build(X.shape)
ours.gain.assign(tf.linspace(0.5, 1.5, 8))
native.set_weights(ours.get_weights())
bool(tf.experimental.numpy.allclose(ours(X), native(X)))

## Layers with Fixed State

Some layers hold arrays that are neither learned nor recomputed: they are
fixed when the layer is built and must stay the same for the life of the
model. An example is a layer of *random features*
[@Rahimi.Recht.2007]. It multiplies a minibatch $\mathbf{X}$ by a
random matrix $\mathbf{W} \in \mathbb{R}^{d \times D}$ with standard normal
entries, adds a random offset $\mathbf{b} \in \mathbb{R}^D$ with entries
uniform on $[0, 2\pi]$, and applies the cosine elementwise:

$$\mathbf{Z} = \sqrt{2/D}\, \cos(\mathbf{X}\mathbf{W} + \mathbf{b}).$$

Each row of $\mathbf{Z}$ holds $D$ nonlinear features of the corresponding
example. Surprisingly, these features need no training: a linear layer on
top of them can fit nonlinear functions of the input while only the linear
layer is trained.

What matters for us is the state. $\mathbf{W}$ and $\mathbf{b}$ are drawn
once and must then not change, so they are not parameters. Yet they must be
saved with the model, since the linear layer on top is trained for these
features and no others. Arrays of this kind are the *buffers* of
that section, saved and moved with the model but never
trained.

In Keras a buffer is a variable created with `add_weight(trainable=False)`.
Each `add_weight` gets an initializer of its own, because one Keras
initializer object returns the same values every time it is called.

In [ ]:
class RandomFeatures(tf.keras.layers.Layer):
    def __init__(self, d, num_features):
        super().__init__()
        normal = tf.keras.initializers.RandomNormal(stddev=1.0)
        uniform = tf.keras.initializers.RandomUniform(0, 2 * math.pi)
        self.W = self.add_weight(name='W', shape=(d, num_features),
                                 initializer=normal, trainable=False)
        self.b = self.add_weight(name='b', shape=(num_features,),
                                 initializer=uniform, trainable=False)

    def call(self, X):
        return math.sqrt(2 / self.b.shape[0]) * tf.cos(X @ self.W + self.b)

Let us see the features at work. We fit $y = \sin(3x)$ at 100 points evenly
spaced in $[-2, 2]$ with a model of two layers, 500 random features followed
by a linear layer, and train it by Adam on the mean squared error. The
optimizer receives the parameters of the whole model, and the buffers are
not among them, so only the linear layer learns:

In [ ]:
x = tf.linspace(-2.0, 2.0, 100)
X, y = tf.reshape(x, (-1, 1)), tf.sin(3 * x)

def mse(net, X, y):
    return tf.reduce_mean((net(X)[:, 0] - y) ** 2)

net = tf.keras.Sequential([RandomFeatures(1, 500), tf.keras.layers.Dense(1)])
optimizer = tf.keras.optimizers.Adam(0.05)
for _ in range(500):
    with tf.GradientTape() as tape:
        loss = mse(net, X, y)
    grads = tape.gradient(loss, net.trainable_variables)
    optimizer.apply_gradients(zip(grads, net.trainable_variables))
print(f'mean squared error {float(mse(net, X, y)):.2g}')
d2l.plot(x, [y, net(X)[:, 0]], 'x', 'y',
         legend=['sin(3x)', 'fit'], figsize=(4.5, 2.5))

The fit follows the sine closely. Its mean squared error ends below 0.01,
against 0.5 for predicting zero everywhere, the mean square of $\sin(3x)$
over the interval.

Now suppose that the checkpoint held only the trained linear layer. A model
rebuilt from it draws a new $\mathbf{W}$ and a new $\mathbf{b}$, and the
linear layer receives features it was never trained on. We load the linear
layer alone into a fresh model, and then the full state, buffers included:

In [ ]:
fresh = tf.keras.Sequential([RandomFeatures(1, 500),
                             tf.keras.layers.Dense(1)])
fresh(X)  # A first call builds the linear layer so its weights exist
fresh.layers[1].set_weights(net.layers[1].get_weights())
print(f'linear layer only: mean squared error '
      f'{float(mse(fresh, X, y)):.2g}')
fresh.set_weights(net.get_weights())
print(f'full state:        mean squared error '
      f'{float(mse(fresh, X, y)):.2g}')

With new random features the error of the trained linear layer rises a
hundredfold or more, to the level of predicting zero or worse. With the
buffers loaded, it is again the error of the trained model.

`net.get_weights()` holds `W` and `b` next to the kernel and bias of the
dense layer, while `net.trainable_variables`, which the optimizer received,
holds only the latter two. Plain tensor attributes would be missing from
both.

The same mechanism holds the fixed tables and running averages of layers in
later chapters, such as the running statistics of batch normalization in
that section.

## Functions with Custom Gradients

So far we wrote forward computations and let automatic differentiation
derive the backward pass. It differentiates the operations we ran,
faithfully, and occasionally that is the problem. Recall the model that is
trained to work with weights stored as 4-bit integers. It rounds its
weights in the forward pass. Rounding is constant between integers, so its
derivative is zero almost everywhere, and any weight behind a rounding
operation receives a zero gradient:

In [ ]:
w = tf.Variable([0.9, 1.4, 2.6])
with tf.GradientTape() as tape:
    y = tf.reduce_sum(tf.round(w))
tape.gradient(y, w, unconnected_gradients=tf.UnconnectedGradients.ZERO)

TensorFlow declares rounding non-differentiable, so without the argument
`unconnected_gradients` the tape returns `None`; the argument asks for the
zeros that calculus gives.

The gradient is zero, as calculus says, and useless for training. The
*straight-through estimator* [@Bengio.Leonard.Courville.2013] replaces
it: the forward pass still rounds, and the backward pass treats the
rounding as if it were the identity, whose derivative is 1. This is
deliberately not the true derivative. Its justification is that the
staircase never strays more than $1/2$ from the identity, as
the figure shows. Automatic differentiation cannot produce such a
rule, because it is not the derivative of what the forward pass computes,
so we supply the backward pass ourselves.

![The straight-through estimator. Forward: the staircase round(x) stays close to the identity (dashed). Backward: the true derivative of rounding is zero almost everywhere; the estimator uses the derivative of the identity, 1, instead.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-ste.svg)

In TensorFlow the tool is the decorator `@tf.custom_gradient`: the
decorated function returns its output together with its backward rule:

In [ ]:
@tf.custom_gradient
def round_ste(X):
    def grad(upstream):
        # Treat the forward pass as the identity: pass the gradient on
        return upstream
    return tf.round(X), grad

Two rules govern the decorator. First, the decorated function returns a
pair: the output and the gradient function. The gradient function is
defined inside the forward pass, so it can use any value that the forward
pass computed, such as the input. Second, `grad` receives the gradient of
the loss with respect to the output and returns one gradient per argument
of the decorated function. A function that reads a `tf.Variable` must also
return gradients for the variables, through a keyword argument `variables`
of `grad`; TensorFlow raises an error that names this requirement if it is
missing.

To check the rule, we multiply the rounded values by known weights
$(1, 2, 3)$ and sum. The straight-through estimator should deliver these
weights, untouched, as the gradient at $\mathbf{w}$:

In [ ]:
with tf.GradientTape() as tape:
    y = tf.reduce_sum(round_ste(w) * tf.constant([1.0, 2.0, 3.0]))
tape.gradient(y, w)

The gradient arrives at $\mathbf{w}$ unchanged, as if the rounding were the
identity, while the forward pass computed with the rounded values. Does a
weight then learn through the rounding? We fit a linear model whose eight
weights are rounded to integers in the forward pass to data generated by
weights drawn uniformly from $[-7, 7]$, the range of 4-bit integers. Gradient
descent starts from $\mathbf{w} = \mathbf{0}$, once with plain rounding and
once with the straight-through estimator. For comparison we also print the
loss at the start and the loss of the generating weights rounded to the
nearest integers:

In [ ]:
X = tf.random.normal((256, 8))
w_true = tf.random.uniform((8,), -7, 7)
y = tf.linalg.matvec(X, w_true)

def fit(rnd, X, y, num_steps=100, lr=0.05):
    w = tf.Variable(tf.zeros(8))
    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            loss = tf.reduce_mean((tf.linalg.matvec(X, rnd(w)) - y) ** 2)
        w.assign_sub(lr * tape.gradient(
            loss, w, unconnected_gradients=tf.UnconnectedGradients.ZERO))
    return float(loss)

rounded_loss = tf.reduce_mean((tf.linalg.matvec(X, tf.round(w_true)) - y) ** 2)
print(f'start {float(tf.reduce_mean(y ** 2)):.3g}, '
      f'rounding {fit(tf.round, X, y):.3g}, '
      f'straight-through {fit(round_ste, X, y):.3g}, '
      f'rounded target {float(rounded_loss):.3g}')

With plain rounding the loss never leaves its starting value, since every
gradient is zero. With the straight-through estimator it falls from tens or
hundreds to about 1, within a small factor of the loss of the rounded
generating weights, which is the error that rounding them would cause. A
gradient that is deliberately wrong has trained the model.

The identity is one choice of surrogate among several. A quantizer to
4-bit integers also clips its input to the representable range, $-8$ to
$7$, and a common variant of the estimator passes the gradient only inside
that range and sets it to zero outside, where the clipped output does not
change (an exercise implements such a rule). The surrogate is a modeling
choice, and the right one depends on the quantizer it stands in for.

For the identity there is a shortcut without a hand-written backward pass.
The papers that introduced the estimator describe it in words; this
shortcut is the form one usually meets in code. A *stop-gradient* operation
is the identity in the forward pass and has derivative zero in the backward
pass, so $X + \textrm{stop}(\textrm{round}(X) - X)$ computes
$\textrm{round}(X)$ in the forward pass while the backward pass sees only
the leading $X$. The general mechanism is still needed when the surrogate
is not the identity, as in the clipped variant. The two definitions agree
in value and in gradient:

In TensorFlow the stop-gradient operation is `tf.stop_gradient`.

In [ ]:
def round_ste2(X):
    return X + tf.stop_gradient(tf.round(X) - X)

def grad_of(f):
    with tf.GradientTape() as tape:
        y = tf.reduce_sum(f(w) * tf.constant([1.0, 2.0, 3.0]))
    return tape.gradient(y, w)

allclose = tf.experimental.numpy.allclose
(bool(allclose(round_ste(w), round_ste2(w))),
 bool(allclose(grad_of(round_ste), grad_of(round_ste2))))

A custom gradient changes the derivative of a computation built from
existing tensor operations; it does not create new operations. Writing new
low-level operations for the GPU is a separate craft that this book does
not cover. that section shows how to get performance out of
the operations that the libraries already provide.

What we have not asked of any of these layers is whether their arithmetic
is accurate. RMSNorm squares its inputs, and the sum of squares can
overflow in low precision. that section takes up the number
formats themselves.

## Summary

A custom layer is a subclass of the module class with a forward method.
Registration does the rest: a learned parameter registered with the module
is found by the optimizer, saved, loaded, and moved like the parameters of
built-in layers, as we checked for the gain of RMSNorm. State that is fixed,
such as the random matrix of a random-feature layer, is registered as a
buffer, which is saved and moved but never trained. When the derivative of
an operation is useless, as for rounding, a custom backward rule such as
the straight-through estimator replaces it. A library implementation is
preferable when it computes the same thing.

In Keras, `add_weight` creates learned state, `add_weight(trainable=False)`
creates buffers, and `@tf.custom_gradient` lets a function return its own
backward rule; `tf.stop_gradient` covers the straight-through case in one
expression.

## Exercises

1. **Centering over the batch.** `CenteredLayer` subtracts the mean over
   every entry of its input, the batch axis included. Consider the network
   of this section, a linear layer with bias $\mathbf{b}$ followed by
   `CenteredLayer`.
    1. Show that the output for one example depends on the other examples
       in the minibatch. State when evaluating the network on one example
       at a time gives the same outputs as evaluating it on the whole
       minibatch at once.
    1. Show that adding the same constant to every entry of $\mathbf{b}$
       leaves the output unchanged, and conclude which direction in the
       space of biases receives zero gradient.
    1. A per-example variant subtracts from each row its own mean over the
       feature axis, as layer normalization does. State which of the two
       properties above this variant keeps.

2. [code] **Scale invariance of RMSNorm.** ● Set $\epsilon = 0$.
    1. Show that $\textrm{RMSNorm}(c\mathbf{x}) = \textrm{RMSNorm}(\mathbf{x})$
       for every $c > 0$.
    1. Differentiate both sides with respect to $c$ at $c = 1$ and conclude
       that the Jacobian $\mathbf{J}$ of RMSNorm at $\mathbf{x}$ satisfies
       $\mathbf{J}\mathbf{x} = \mathbf{0}$.
    1. Conclude that the gradient of any loss with respect to the input
       $\mathbf{x}$ of RMSNorm is orthogonal to $\mathbf{x}$, and check
       this with automatic differentiation for a random $\mathbf{x}$ and a
       random linear loss. What does a small gradient step do to the
       length of $\mathbf{x}$?

3. [code] **RMSNorm bias.** Add an optional learned bias $\mathbf{b}$ to
   `RMSNorm` (a shift applied after the gain, restoring part of what layer
   normalization had), and verify that `get_weights` on a model containing
   it grows by the expected entry. Then place the layer directly before a
   linear layer with weight $\mathbf{W}$ and bias $\mathbf{c}$, and show
   that $\mathbf{b}$ can be folded into $\mathbf{c}$ without changing the
   outputs. What is the folded bias?

4. [code] **Random features approximate a kernel.** Let $\mathbf{w}$ have
   independent standard normal entries and $b$ be uniform on $[0, 2\pi]$.
    1. Using
       $\cos\alpha\cos\beta = \frac{1}{2}(\cos(\alpha - \beta) + \cos(\alpha + \beta))$
       and $\mathbb{E}[\cos(\mathbf{w}^\top\mathbf{u})] = \exp(-\|\mathbf{u}\|^2/2)$,
       show that the features $\mathbf{z}(\mathbf{x})$ of `RandomFeatures`
       satisfy
       $\mathbb{E}[\mathbf{z}(\mathbf{x})^\top \mathbf{z}(\mathbf{x}')] = \exp(-\|\mathbf{x} - \mathbf{x}'\|^2/2)$
       for every number $D$ of features.
    1. Check this numerically for two inputs at distance 1, with $D = 10$,
       100, 1,000, and 10,000, averaging the error over several layers for
       each $D$. How does the error shrink as $D$ grows?
    1. Repeat the fit of $\sin(3x)$ from this section with $D = 5$ and
       $D = 50$ features, and with a linear model on $x$ alone in place of
       the random features. What error does each reach, and why?

    *Adapted from [@Rahimi.Recht.2007].*

5. [code] **A clamp with a custom gradient.** Implement a
   `@tf.custom_gradient` function whose forward is `tf.clip_by_value(X, lo,
   hi)` and whose backward passes the gradient only where the input lay
   strictly inside the clamp range; the gradient closure must capture `X`.
   Compare its gradients with those of the native `tf.clip_by_value` for
   inputs inside, outside, and exactly on the boundary. Which convention
   does the native operation use at the boundary?

    *Adapted from the PyTorch tutorial
    [Learning PyTorch with Examples](https://docs.pytorch.org/tutorials/beginner/pytorch_with_examples.html)
    and the JAX notebook
    [Custom derivative rules](https://docs.jax.dev/en/latest/notebooks/Custom_derivative_rules_for_Python_code.html).*

6. [extended] **Bucketing with a straight-through estimator.** ●
   The experiment of this section rounded the weights themselves. Here the
   rounding sits between two layers, on an activation. Write a layer whose forward pass maps a score $s$ to the
   bucket index $\lfloor K\sigma(s) \rfloor$ and whose backward pass treats
   the floor as the identity. Return the index as a floating-point tensor so
   that gradients can reach $s$. Place the layer with $K = 4$ between two
   linear layers of output width one, and train the network on inputs
   $\mathbf{x} \in \mathbb{R}^{10}$ with targets
   $y = \lfloor 4\sigma(\mathbf{w}^\top \mathbf{x}) \rfloor$ for a fixed
   random $\mathbf{w}$. Compare the final training loss with that of the
   same network in which the floor keeps its true gradient, and explain
   which parameters learn in each case.

[Discussions](https://d2l.discourse.group/t/36755)